# 2.5 Vector 计算指令：FA 的 Softmax 引擎

FA 结构中“两次 Cube 夹一次 Vector”的那段 Vector——Online Softmax 的全部递推——就靠本节的向量指令完成。第五块积木，也是 950 硬件升级（双发射）的直接受益者。

## 1. 执行模型：一条指令，一队数据

Vector Core 的 SIMD 模式：**一条指令同时作用于一段连续数据**（UB 中的 128B/256B 对齐段），如一条 `Exp` 指令对 64 个 FP16 同时求指数。这与 Scalar“一次一个数”形成数量级差异。（950 的双发射与 SIMT 模式已在 [1.2 节](01.02_ascend_950_arch.ipynb)展开，此处不重复。）

此外，Vector 侧还配有 **SFU（Special Function Unit）**：把 sin/cos 这类难以并行的特殊函数硬化成单条指令执行——一指令一功能，用专用电路换通用流水线的效率。

## 2. 向量指令三要素：Repeat / Mask / Stride

向量指令的“操作范围”由三个参数刻画：

<img src="./images/vector_api_characteristics.png" width="600">

> **看图要点**：
> - **Repeat**：指令重复次数——一段 256B 数据为一次基本操作，处理更长数据就重复多次；
> - **Mask**：**位掩码控制哪些元素参与运算**（如 KV 尾块不足 256B 时，用 mask 屏蔽无效位置）——FA 尾块处理的必备工具；
> - **Stride**：源/目的的地址步长——支持“隔行取数”等非连续访问。

| 要素 | 解决的问题 | FA 中的用武之地 |
|--|--|--|
| Repeat | 数据比一次操作长怎么办 | 整行 Softmax 循环处理 |
| Mask | 边界不齐怎么办 | KV 尾块、Causal 部分掩码 |
| Stride | 数据不连续怎么办 | 分块间的跨步读写 |

> **Mask 是 FA 的老朋友**：第2章的 Causal 掩码块跳过、第3篇的尾块处理，落到硬件上全是“给向量指令配一个正确的 mask”。

## 3. 常用向量指令速查（以 FA 视角）

| Ascend C API | 数学含义 | Online Softmax 递推中的角色 |
|--|--|--|
| `Exp(x)` | 逐元素指数 | e^(S - m_new) |
| `Max(x, y)` / `ReduceMax` | 逐元素取大 / 行归约最大 | m_new = max(m_old, row_max(S)) |
| `ReduceSum` | 行归约求和 | l_new = l_old·e^(m_old-m_new) + row_sum |
| `Mul` / `Add` | 逐元素乘 / 加 | rescale 旧 O、更新累加 |
| `Sub` | 逐元素减 | S - m（数值稳定） |
| `Brcb` / 广播类 | 标量→向量广播 | 行 max/sum 广播到整行 |

> 对照第2章 Online Softmax 三步递推公式，逐行检查这张表——**你会发现整个递推就是六七条向量指令的组合**。第3篇第 10 讲（在线 Softmax 模块）将把它们写成真实代码。

## 4. API 分级：从便捷到极致

同一类计算，Ascend C 提供 4 个层级的 API，封装程度递减、可控性递增：

| 级别 | 形态 | 示例 | 一句话 |
|--|--|--|--|
| 3 级 | 运算符重载 | `dst = src1 + src2` | 最直观，适合原型验证 |
| 2 级 | 连续 count 计算 | `Add(dst, src1, src2, count)` | 一次指定一维连续段 |
| 1 级 | 多维 slice 计算（演进中） | 对 Tensor 切片直接运算 | 多维局部计算 |
| 0 级 | 完全参数控制 | `Add(dst, src1, src2, repeatTimes, repeatParam)` | repeat/mask/stride 全开放，榨干硬件 |

> 上一节的**三要素（Repeat/Mask/Stride），正是 0 级 API 的参数面板**。开发路径通常是：先写 3 级验证正确性，再把性能热点下沉到 2 级 / 0 级。FA Kernel 属于性能敏感型，第3篇会直接以 2 级 / 0 级风格书写——这也是本节把三要素讲透的原因。

## 5. Vector 编程范式：三段式

<img src="./images/vector_programming_paradigm.png" width="600">

> **看图要点**：**CopyIn → Compute → CopyOut**（比 Cube 少了 Split/Aggregate，因为 UB 不要求分形）。在 Ascend C 的队列 API 里，这三段天然映射为 `inQueue.DeQue()` → 计算 → `outQueue.EnQue()`（队列细节见下一节 2.6）。

**FA 的 Vector 段落**就嵌在两个 Cube 段之间：S 从 L0C 进 UB（950 可直连），三段式跑完 Online Softmax，产出的 P 再供下一次矩阵乘——上一节与下一节的积木在此咬合。

## 小结

| 关键词 | 一句话 |
|--|--|
| SIMD | 一条指令处理一段对齐数据 |
| 三要素 | Repeat（多长）、Mask（哪些）、Stride（怎么走） |
| FA 映射 | Online Softmax ≈ 6~7 条向量指令的组合 |
| 三段式 | CopyIn → Compute → CopyOut，无分形税 |
| API 分级 | 3 级便捷 → 0 级极致，三要素就是 0 级参数面板 |

最后一块积木：[2.6 同步指令](01.08_sync_instructions.ipynb)——让前五块积木严丝合缝地咬合。

## 课后练习

1.（单选）KV 尾块长度不是指令对齐长度整数倍时，应主要依靠哪个参数屏蔽无效元素？（A. Repeat　B. Mask　C. Stride　D. EnQue）

2.（判断）Vector 算子与 Cube 算子的编程范式段数相同。

3.（单选）Online Softmax 递推中的 rescale（旧结果乘 e^(m_old-m_new)）用哪类指令最合适？（A. mmad　B. MTE2　C. 向量乘法 Mul　D. SetFlag）

> 完成后查看 [answer/01.07_answer.txt](answer/01.07_answer.txt) 核对答案。